# HW01-B — SQL, Latency, and Metabase

The business team does not care that your notebook works. They want a dashboard that opens fast.

Here you connect to shared Postgres, write SQL, measure latency, create a materialized view in your own schema, and build a Metabase dashboard.


## Submission discipline

This is individual work.

Work locally. Push to GitHub. Use the shared server services through URLs and credentials. Do not SSH into the server.

Do not commit `.env`, `.venv/`, passwords.


## Credentials and shared services

Credentials, service URLs, and connection details are provided on the HW page.

Use those exact values. Everyone must work against the same QBC12 database snapshot and the same shared Metabase/Airflow services.

Do not paste credentials into notebook markdown. Do not commit `.env` files. Do not screenshot passwords.


## Useful references

- PostgreSQL `EXPLAIN`: https://www.postgresql.org/docs/current/sql-explain.html
- PostgreSQL using `EXPLAIN`: https://www.postgresql.org/docs/current/using-explain.html
- Metabase questions: https://www.metabase.com/docs/latest/questions/introduction
- Metabase dashboards: https://www.metabase.com/docs/latest/dashboards/introduction

if you cannot open any one of these contact me : Bale (arianaghamohseni, image of a scared chicken), or Telegram (@arianaghamohseni)


## What to avoid

- `select *` in dashboard queries.
- Creating objects in `core`. You do not own `core`.
- Optimizing without runtime measurements.
- Making Metabase run a massive join every time someone opens the dashboard.


In [1]:
import os, re, time
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

load_dotenv()

for path in ["sql", "reports", "screenshots"]:
    Path(path).mkdir(exist_ok=True)

DB_HOST = os.getenv("QBC12_DB_HOST", "SERVERIP")  # this is in the excel file give in Quera
DB_PORT = os.getenv("QBC12_DB_PORT", "32112")
DB_NAME = os.getenv("QBC12_DB_NAME", "qbc12_airbnb")
DB_USER = os.getenv("QBC12_DB_USER", "") or input("DB user: ").strip()
DB_PASSWORD = os.getenv("QBC12_DB_PASSWORD", "") or input("DB password: ").strip()
STUDENT_ID = os.getenv("QBC12_STUDENT_ID", "") or DB_USER.replace("student_", "")

safe_student = re.sub(r"[^a-zA-Z0-9_]", "_", STUDENT_ID.lower())
STUDENT_SCHEMA = f"student_{safe_student}"
engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}", pool_pre_ping=True
)
with engine.begin() as conn:
    conn.execute(text("SET statement_timeout = '30s'"))
    version = conn.execute(text("select version()")).scalar()
STUDENT_SCHEMA, version[:80]

('student_amirhossein_sa',
 'PostgreSQL 16.14 (Debian 16.14-1.pgdg13+1) on x86_64-pc-linux-gnu, compiled by g')

## 1. Inspect before querying

You are not allowed to write the final query blind. Check columns and row counts first.


In [3]:
columns_sql = """
select table_schema, table_name, column_name, data_type
from information_schema.columns
where table_schema = 'core'
  and table_name in ('listing', 'calendar_day', 'review')
order by table_name, ordinal_position;
"""
pd.read_sql(columns_sql, engine)

,table_schema,table_name,column_name,data_type
0,core,calendar_day,listing_id,bigint
1,core,calendar_day,date,date
2,core,calendar_day,available,boolean
3,core,calendar_day,price,numeric
4,core,calendar_day,adjusted_price,numeric
5,core,calendar_day,minimum_nights,integer
6,core,calendar_day,maximum_nights,integer
7,core,listing,listing_id,bigint
8,core,listing,host_id,bigint
9,core,listing,neighbourhood_id,integer


In [3]:
row_count_sql = """
select 'core.listing' as table_name, count(*) as rows from core.listing
union all select 'core.calendar_day', count(*) from core.calendar_day
union all select 'core.review', count(*) from core.review;
"""
pd.read_sql(row_count_sql, engine)

,table_name,rows
0,core.listing,10480
1,core.calendar_day,3825200
2,core.review,501084


## 2. Create your sandbox schema

This is the only place you write database objects.


In [4]:
# TODO 2.1
# Create your schema if it does not exist.
# Schema name is STUDENT_SCHEMA.

with engine.begin() as conn:
    with engine.begin() as conn:
        schema_exists = conn.scalar(
            text(
                "select 1 from information_schema.schemata "
                "where schema_name = :schema"
            ),
            {"schema": STUDENT_SCHEMA},
        )
        print("schema exists:", schema_exists)
        if not schema_exists:
            conn.execute(text(f'CREATE SCHEMA "{STUDENT_SCHEMA}" AUTHORIZATION CURRENT_USER'))

schema exists: 1


## 3. Build baseline SQL in pieces

Do not write one giant query first. Build the CTEs, test them, then combine.


In [4]:
# TODO 3.1
# Write calendar_30_sql.
# Required output: listing_id, avg_calendar_price_30, availability_30_rate.

calendar_30_sql = '''
with date_bounds as (
    select min(date) as start_date
    from core.calendar_day
),
calendar_30 as (
    select
        cd.listing_id,
        coalesce(cd.price, l.listing_price) as calendar_price,
        cd.available
    from core.calendar_day cd
    join core.listing l
      on l.listing_id = cd.listing_id
    join date_bounds db
      on cd.date >= db.start_date
     and cd.date < db.start_date + interval '30 days'
),
calendar_30_agg as (
    select
        listing_id,
        round(avg(calendar_price), 2) as avg_calendar_price_30,
        round(avg(available::int)::numeric, 4) as availability_30_rate
    from calendar_30
    group by listing_id
)
select
    listing_id,
    avg_calendar_price_30,
    availability_30_rate
from calendar_30_agg
'''

pd.read_sql(calendar_30_sql + "limit 10", engine)

,listing_id,avg_calendar_price_30,availability_30_rate
0,27886,132.0,0.0667
1,28871,89.0,0.0000
2,29051,61.0,0.0000
3,44391,NaN,0.0000
4,48373,NaN,0.0000
5,49552,322.0,0.2000
6,50263,457.0,0.7333
7,50515,198.0,0.0000
8,50523,162.0,0.0333
9,53921,NaN,0.0000


In [ ]:
# TODO 3.2
# Write review_counts_sql.
# Required output: listing_id, total_reviews.

review_counts_sql = """
-- write SQL here
"""

In [ ]:
# TODO 3.3
# Combine the CTEs with core.listing into baseline_sql.
# Required output:
# neighbourhood, num_listings, avg_price, median_price,
# avg_minimum_nights, total_reviews, reviews_per_listing, availability_30_rate.

baseline_sql = """
-- write SQL here
"""
Path("sql/01_baseline_neighbourhood_summary.sql").write_text(baseline_sql)

In [ ]:
def timed_read_sql(sql: str, repeats: int = 3):
    times = []
    last_df = None
    for _ in range(repeats):
        start = time.perf_counter()
        last_df = pd.read_sql(sql, engine)
        times.append(time.perf_counter() - start)
    return last_df, times


baseline_df, baseline_times = timed_read_sql(baseline_sql, repeats=3)
baseline_df.head(), baseline_times

## 4. Read the query plan

`EXPLAIN ANALYZE` actually runs the query. Look for big scans, expensive joins, and repeated work.


In [ ]:
# TODO 4.1
# Run EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT) on baseline_sql.
# Save the plan to reports/baseline_explain_analyze.txt.

# Write your code here.

In [ ]:
# TODO 4.2
# Write reports/explain_notes.md with 3 specific observations from the plan.
# Do not write vague nonsense like 'the query is slow'.

# Write your code here.

## 5. Create a materialized view

Metabase should read from a prepared object, not a fresh monster join.


In [ ]:
# TODO 5.1
# Create optimized_sql.
# It should create student_<you>.mv_airbnb_neighbourhood_summary and at least two indexes.

optimized_sql = f"""
-- write SQL here
"""
Path("sql/02_create_materialized_view.sql").write_text(optimized_sql)

In [ ]:
# TODO 5.2
# Execute optimized_sql statement by statement.

# Write your code here.

In [ ]:
check_sql = (
    f"""select * from "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary order by num_listings desc limit 10;"""
)
pd.read_sql(check_sql, engine)

## 6. Compare latency

Numbers or it did not happen.


In [ ]:
dashboard_sql = f"""
select neighbourhood, num_listings, avg_price, median_price,
       total_reviews, reviews_per_listing, availability_30_rate, availability_365_rate
from "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary
order by num_listings desc;
"""
dashboard_df, dashboard_times = timed_read_sql(dashboard_sql, repeats=5)
perf = pd.DataFrame(
    [
        {
            "query": "baseline_direct_query",
            "best_seconds": min(baseline_times),
            "avg_seconds": sum(baseline_times) / len(baseline_times),
        },
        {
            "query": "materialized_view_read",
            "best_seconds": min(dashboard_times),
            "avg_seconds": sum(dashboard_times) / len(dashboard_times),
        },
    ]
)
perf["speedup_vs_baseline_best"] = perf.loc[0, "best_seconds"] / perf["best_seconds"]
perf

## 7. Metabase dashboard

Open the shared Metabase URL and create:

```text
QBC12 HW01 - <your-github-username> - Airbnb Ops
```

Required cards:

1. listings by neighbourhood
2. average price by neighbourhood
3. review activity by neighbourhood
4. availability rate by neighbourhood
5. top neighbourhoods table

Screenshot path:

```text
screenshots/metabase_dashboard.png
```


In [ ]:
# TODO 7.1
# Write reports/hw01_b_sql_performance.md.
# Include schema, runtimes, speedup, what changed, and Metabase screenshot/link.

# Write your code here.

In [ ]:
for file in [
    "sql/01_baseline_neighbourhood_summary.sql",
    "sql/02_create_materialized_view.sql",
    "reports/baseline_explain_analyze.txt",
    "reports/explain_notes.md",
    "reports/hw01_b_sql_performance.md",
]:
    assert Path(file).exists(), f"Missing {file}"
assert len(dashboard_df) > 0
perf

## Commit

```bash
git add sql reports screenshots notebooks
git commit -m "HW01-B SQL performance and Metabase dashboard"
```
